# Fine Tuning SAM Audio

```mermaid
flowchart TD
    subgraph Data Preparation Phase
        RawStems[Raw Instrument Stems Baglama and Zurna] --> DynamicMix[Dynamic Mixing Strategy]
        DynamicMix --> MixtureAudio[Complex Mixture Audio]
        DynamicMix --> TargetAudio[Isolated Ground Truth Audio]
        Prompt[Text Prompt e g baglama] --> Tuple[Create Training Tuple]
        MixtureAudio --> Tuple
        TargetAudio --> Tuple
    end

    subgraph Latent Compression Phase DAC-VAE
        Tuple --> LoadVAE[Load Pre-trained DAC-VAE]
        LoadVAE --> FreezeVAE[Freeze VAE Weights No Gradients]
        FreezeVAE --> EncodeLatents[Encode Audio to Continuous Latents]
        EncodeLatents --> LatentTensors[Output 40Hz 64-dimensional pt Tensors]
    end

    subgraph SimpleTuner Configuration Phase
        LatentTensors --> JSONL[Format metadata jsonl dataset map]
        JSONL --> Dropout[Configure Prompt Dropout Regularization]
        Dropout --> ConfigArgs[Set Command Line Arguments]
    end

    subgraph LoRA Flow-Matching Training Phase
        ConfigArgs --> LoadBase[Load Frozen SAM Audio Large Base]
        LoadBase --> InjectLoRA[Inject LoRA Adapters]
        InjectLoRA --> TargetLayers[Target q_proj and v_proj Cross-Attention]
        TargetLayers --> VectorField[Execute Flow-Matching Training Loop]
        VectorField --> SaveAdapter[Export adapter_model safetensors]
    end

    subgraph Inference and Validation Phase
        SaveAdapter --> DeployModel[Load Base Model and New Adapter]
        DeployModel --> InputNovel[Input Novel Unseen Audio Mixture]
        InputNovel --> DiTSeparation[DiT Generates Target Latents]
        DiTSeparation --> DecodeAudio[DAC-VAE Decoder Reconstructs Audio]
    end
```

## Data Preparation

## Latent Compression

import os
import pandas as pd
import torch
import torchaudio
from dacvae import DACVAE
from tqdm.auto import tqdm

# Load metadata
metadata_path = '../data/mixed/metadata.csv'
df = pd.read_csv(metadata_path)

# Initialize paths for latents
dirs = {
    'mixed': '../data/latents/mixed/',
    'baglama': '../data/latents/baglama/',
    'zurna': '../data/latents/zurna/'
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

# Load frozen VAE model
print("Loading DAC-VAE model...")
model = DACVAE.load("facebook/dacvae-watermarked")
model.eval() # Freeze model
for param in model.parameters():
    param.requires_grad = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def process_audio(file_path, model_sample_rate):
    try:
        wav, sample_rate = torchaudio.load(file_path)
        # Resample
        if sample_rate != model_sample_rate:
            wav = torchaudio.functional.resample(wav, sample_rate, model_sample_rate)
        # Convert stereo to mono if applicable
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        # Expected shape is batch x 1 x samples
        wav = wav.unsqueeze(0).to(device)
        return wav
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

print("Encoding audio to latents...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    mixed_file = row['file_name']
    zurna_source = row['zurna_source']
    baglama_source = row['baglama_source']

    # Resolve paths
    mixed_path = os.path.join('../data/mixed/', mixed_file)
    zurna_path = os.path.join('../data/processed/zurna/', zurna_source)
    baglama_path = os.path.join('../data/processed/baglama/', baglama_source)

    # Process and encode each category
    file_mappings = [
        ('mixed', mixed_path, mixed_file), 
        ('zurna', zurna_path, zurna_source), 
        ('baglama', baglama_path, baglama_source)
    ]
    
    for label, audio_path, filename in file_mappings:
        base_name = filename.replace('.wav', '.pt')
        out_path = os.path.join(dirs[label], base_name)
        
        # Skip if already encoded
        if os.path.exists(out_path):
            continue
            
        wav_input = process_audio(audio_path, model.sample_rate)
        if wav_input is not None:
            with torch.no_grad():
                encoded = model.encode(wav_input)
            torch.save(encoded.cpu(), out_path)
            
print("Latent encoding complete!")